# 🎭 ChuckleNet: Process Local Files (FIXED PATH)

**Upload your files to Google Drive FIRST:**
```bash
# On your LOCAL machine:
rclone copy /Users/Subho/data/utterances/vtt_audio_local/ gdrive:chuckle_net/audio/ --progress
rclone copy /Users/Subho/data/chuckle_vtt_labels/ gdrive:chuckle_net/vtt/ --progress
```

**This notebook:**
- Parses VTT → utterance timestamps + [laughter] markers
- Extracts F0 at utterance boundaries (NOT 1s fixed)
- Labels by [laughter] overlap
- Trains F0 model
- Reports multilingual F1

In [ ]:
# 1. Setup & Mount
from google.colab import drive
drive.mount('/content/drive')

!pip install librosa numpy pandas scikit-learn tqdm 2>&1 | tail -3

import os, glob
import numpy as np
from tqdm import tqdm

# Try different base paths (Colab mounts Drive differently)
for base_candidate in [
    '/content/drive/My Drive/chuckle_net',
    '/content/drive/Shareddrives/chuckle_net',
    '/content/drive/MyDrive/chuckle_net',
    '/content/drive/Shared drives/chuckle_net',
]:
    if os.path.exists(base_candidate):
        BASE = base_candidate
        print(f'✅ Found: {BASE}')
        break
else:
    BASE = '/content/drive/My Drive/chuckle_net'
    print(f'⚠️ Using default: {BASE}')

AUDIO_DIR = f'{BASE}/audio'
VTT_DIR = f'{BASE}/vtt'

print(f'Audio dir: {AUDIO_DIR}')
print(f'VTT dir: {VTT_DIR}')

# List files
audio_files = glob.glob(f'{AUDIO_DIR}/*.wav') + glob.glob(f'{AUDIO_DIR}/*.m4a') + glob.glob(f'{AUDIO_DIR}/*.mp3')
vtt_files = glob.glob(f'{VTT_DIR}/*.vtt')

print(f'Audio files found: {len(audio_files)}')
print(f'VTT files found: {len(vtt_files)}')

# Show samples
if audio_files:
    print(f'Sample audio: {os.path.basename(audio_files[0])}')
if vtt_files:
    print(f'Sample VTT: {os.path.basename(vtt_files[0])}')

In [ ]:
# 2. Build audio → VTT mapping

def get_video_id(filename):
    """Extract video ID from audio or VTT filename."""
    # audio: 0-LpztqlymY.m4a → 0-LpztqlymY
    # VTT: 0-LpztqlymY.en.vtt → 0-LpztqlymY
    name = os.path.basename(filename)
    vid = name.replace('.en.vtt','').replace('.vtt','').replace('.m4a','').replace('.wav','').replace('.mp3','')
    return vid

# Build VTT lookup
vtt_lookup = {}  # vid → vtt_path
for vtt in vtt_files:
    vid = get_video_id(vtt)
    vtt_lookup[vid] = vtt

# Build audio lookup
audio_lookup = {}  # vid → audio_path
for aud in audio_files:
    vid = get_video_id(aud)
    audio_lookup[vid] = aud

# Find matching pairs
matching_vids = set(audio_lookup.keys()) & set(vtt_lookup.keys())
print(f'Audio files: {len(audio_lookup)}')
print(f'VTT files: {len(vtt_lookup)}')
print(f'Matching pairs: {len(matching_vids)}')

# Show sample matches
sample_vids = list(matching_vids)[:3]
for vid in sample_vids:
    print(f'  {vid}:')
    print(f'    audio: {os.path.basename(audio_lookup[vid])}')
    print(f'    VTT: {os.path.basename(vtt_lookup[vid])}')

In [ ]:
# 3. Parse VTT and extract F0
import librosa, re

def parse_vtt_cues(vtt_path):
    """Parse VTT, return list of (start, end, text, has_laughter)."""
    with open(vtt_path, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    
    def to_sec(ts):
        ts = ts.replace('.', ':')
        p = ts.split(':')
        return int(p[0])*3600 + int(p[1])*60 + float(p[2])
    
    cues = []
    lines = content.split('\n')
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if '-->' in line:
            start, end = line.split('-->')
            start, end = to_sec(start.strip()), to_sec(end.strip())
            text_lines = []
            i += 1
            while i < len(lines) and lines[i].strip() and '-->' not in lines[i]:
                text_lines.append(lines[i].strip())
                i += 1
            text = ' '.join(text_lines)
            has_laughter = '[laughter]' in text.lower()
            cues.append((start, end, text, has_laughter))
        else:
            i += 1
    return cues

def extract_f0_features(y, sr=22050):
    """Extract 5-dim F0 features."""
    try:
        f0, _, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr, hop_length=512)
        f0 = np.nan_to_num(f0, nan=0.0)
        return [
            float(np.mean(f0)), float(np.std(f0)),
            float(np.max(f0)), float(np.min(f0)),
            float(np.mean(f0 > 0))
        ]
    except:
        return [0, 0, 0, 0, 0]

# Test on one pair
if matching_vids:
    vid = list(matching_vids)[0]
    audio_path = audio_lookup[vid]
    vtt_path = vtt_lookup[vid]
    print(f'Testing with: {vid}')
    print(f'  Audio: {os.path.basename(audio_path)}')
    print(f'  VTT: {os.path.basename(vtt_path)}')
    
    cues = parse_vtt_cues(vtt_path)
    print(f'  VTT cues: {len(cues)}')
    print(f'  Laughter cues: {sum(1 for c in cues if c[3])}')
    
    try:
        y, sr = librosa.load(audio_path, sr=22050)
        print(f'  Audio loaded: {len(y)/sr:.1f}s')
    except Exception as e:
        print(f'  Audio load error: {e}')

In [ ]:
# 4. Batch Process All Matching Videos
all_features, all_labels, all_vids, all_uids, all_langs = [], [], [], [], []

matching_list = list(matching_vids)
print(f'Processing {len(matching_list)} videos...')

for vid in tqdm(matching_list, desc='Processing videos'):
    audio_path = audio_lookup[vid]
    vtt_path = vtt_lookup[vid]
    
    # Detect language from VTT filename
    lang = 'en'
    if '.hi.' in vtt_path: lang = 'hi'
    elif '.zh.' in vtt_path: lang = 'zh'
    elif '.es.' in vtt_path: lang = 'es'
    
    try:
        # Parse VTT
        cues = parse_vtt_cues(vtt_path)
        if not cues:
            continue
        
        # Load audio
        y, sr = librosa.load(audio_path, sr=22050)
        
        # Process each cue
        for i, (start, end, text, has_laughter) in enumerate(cues):
            if end <= start or end - start > 30:
                continue
            
            y_seg = y[int(start*sr):int(end*sr)]
            if len(y_seg) < sr * 0.1:
                continue
            
            feat = extract_f0_features(y_seg, sr)
            all_features.append(feat)
            all_labels.append(1 if has_laughter else 0)
            all_vids.append(vid)
            all_uids.append(f'{vid}_{i}')
            all_langs.append(lang)
            
    except Exception as e:
        # Skip problematic videos
        continue

X = np.array(all_features)
y = np.array(all_labels)
vids = np.array(all_vids)
uids = np.array(all_uids)
langs = np.array(all_langs)

print(f'\n✅ Total: {len(X)} segments from {len(set(vids))} videos')
print(f'Positive: {y.sum()} ({100*y.mean():.1f}%)')

for lang in ['en', 'hi', 'zh', 'es']:
    mask = langs == lang
    if mask.sum() > 0:
        print(f'  {lang}: {mask.sum()} segs, {100*y[mask].mean():.1f}% positive')

In [ ]:
# 5. Train + Evaluate (Video-level split)
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import f1_score, precision_score, recall_score

X_train, X_test, y_train, y_test = [], [], [], []

unique_vids = list(set(vids))
np.random.seed(42)
np.random.shuffle(unique_vids)
n_test = max(1, int(len(unique_vids) * 0.2))
test_vids = set(unique_vids[:n_test])

for i, vid in enumerate(vids):
    if vid in test_vids:
        X_test.append(X[i])
        y_test.append(y[i])
    else:
        X_train.append(X[i])
        y_train.append(y[i])

X_train, X_test = np.array(X_train), np.array(X_test)
y_train, y_test = np.array(y_train), np.array(y_test)

print(f'Train: {len(X_train)} ({100*y_train.mean():.1f}% positive)')
print(f'Test:  {len(X_test)} ({100*y_test.mean():.1f}% positive)')

# LR
print('\n=== Logistic Regression ===')
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)
print(f'F1: {f1_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall: {recall_score(y_test, y_pred):.4f}')

# MLP
print('\n=== MLP ===')
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
mlp.fit(X_train, y_train)
y_pred_mlp = mlp.predict(X_test)
print(f'F1: {f1_score(y_test, y_pred_mlp):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_mlp):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_mlp):.4f}')

In [ ]:
# 6. Save
import pickle

out = {
    'features': X,
    'labels': y,
    'vids': vids,
    'uids': uids,
    'langs': langs
}
np.savez_compressed(f'{BASE}/processed_local_620.npz', **out)
print(f'Saved: {BASE}/processed_local_620.npz')

with open(f'{BASE}/f0_model_local.pkl', 'wb') as f:
    pickle.dump(lr, f)
print(f'Model: {BASE}/f0_model_local.pkl')

print('\n✅ DONE!')